# Auditory Oddball ERP from CSV using MNE

This notebook loads an LSL-recorded CSV file, constructs an MNE Raw object, detects events, epochs the data, and plots ERPs for the auditory oddball paradigm.

Notes:
- It attempts to auto-detect the time column, EEG channel columns, and event/marker column.
- If your CSV uses different column names, tweak the parameters in the next cell.
- MNE expects EEG channel data in Volts. If your CSV is in microvolts (common), data will be scaled to Volts automatically.

In [26]:
# Imports (installs MNE if missing)
import sys, subprocess
import mne
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib qt
matplotlib.style.use('default')

## Parameters
Update the CSV path if needed and adjust the configuration hints if auto-detection fails.

In [ ]:
# Path to your CSV file
csv_path = r"c:\Users\Admin\.eegnb\data\auditory_oddball\local\lsl\subject0001\session000\recording_2025-09-11-03.18.14.csv"

# Column hints for this LSL data format
time_col_hint = "timestamps"  # LSL uses 'timestamps' column
event_col_hint = 'stim'       # LSL uses 'stim' column for markers

# Event mapping for auditory oddball paradigm
# Based on your data: 0=no stimulus, 1=standard tone, 2=target tone
event_map_hint = {
    'Standard': 1,  # Standard/frequent tone
    'Target': 2     # Target/deviant tone
}

# Epoching parameters
tmin, tmax = -0.2, 0.8        # Time window around events (200ms before to 800ms after)
baseline = (None, 0.0)        # Baseline from start of epoch to stimulus onset
l_freq, h_freq = 1.0, 30.0    # Band-pass filter for ERP analysis
notch = 60.0                  # Notch filter for 60Hz line noise (US standard)

## Load CSV and detect columns
This will try to detect the time column, event/marker column, and EEG channels.

In [ ]:
df = pd.read_csv(csv_path)
print('Dataset info:')
print(f'Shape: {df.shape}')
print('Columns:', list(df.columns))

# Detect time column
if time_col_hint and time_col_hint in df.columns:
    time_col = time_col_hint
else:
    candidates = [c for c in df.columns if c.lower() in ['timestamps', 'timestamp', 'time', 'ts', 't']]
    time_col = candidates[0] if candidates else None
print('Time column:', time_col)

# Detect event/marker column
if event_col_hint and event_col_hint in df.columns:
    event_col = event_col_hint
else:
    evt_candidates = [c for c in df.columns if c.lower() in ['marker', 'event', 'stim', 'trigger', 'markervalue']]
    # Also consider low-cardinality integer/object columns as potential event channels
    for c in df.columns:
        if c == time_col:
            continue
        ser = df[c]
        nunique = ser.nunique(dropna=True)
        if nunique and nunique < max(10, int(0.01 * len(ser))):
            evt_candidates.append(c)
    # Keep stable order and uniqueness
    seen = set()
    evt_candidates = [x for x in evt_candidates if not (x in seen or seen.add(x))]
    event_col = evt_candidates[0] if evt_candidates else None
print('Event column:', event_col)

# Show event/stimulus information
if event_col is not None:
    print(f'Event column "{event_col}" unique values:', sorted(df[event_col].unique()))
    print('Event value counts:')
    print(df[event_col].value_counts().sort_index())

# Determine EEG channel columns: numeric columns excluding time/event
exclude = set([c for c in [time_col, event_col] if c is not None])
numeric_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
# Heuristic: drop common non-EEG numeric channels (like counter, sample index) by name
non_eeg_like = set([c for c in numeric_cols if c.lower() in ['counter', 'sample', 'index']])
eeg_cols = [c for c in numeric_cols if c not in non_eeg_like]
print('EEG channels:', eeg_cols)
assert len(eeg_cols) > 0, 'No EEG channel columns detected; please set eeg_cols manually.'

# Extract times and estimate sampling frequency using duration-based method
if time_col is not None:
    times = df[time_col].to_numpy()
    # Handle Unix timestamps (convert to relative time in seconds)
    if times[0] > 1e9:  # Unix timestamp detected
        times_rel = times - times[0]  # Make relative to start
        print(f'Unix timestamps detected, converted to relative time')
        print(f'Recording duration: {times_rel[-1]:.2f} seconds')
    else:
        times_rel = times
    
    # Calculate sampling frequency using duration-based method
    total_duration = times_rel[-1] - times_rel[0]
    total_samples = len(times_rel)
    sfreq = (total_samples - 1) / total_duration if total_duration > 0 else 256.0
    print(f'Duration-based sampling rate calculation:')
    print(f'  Total duration: {total_duration:.3f} seconds')
    print(f'  Total samples: {total_samples}')
    print(f'  Sampling frequency: {sfreq:.2f} Hz')
    
else:
    times = None
    times_rel = None
    sfreq = 256.0  # fallback if no time column
    print('No time column found, using fallback sampling frequency: 256 Hz')

# Data matrix (n_channels, n_times) 
data = df[eeg_cols].to_numpy(dtype=float).T
n_ch, n_times = data.shape
print(f'Data shape (n_channels, n_times): {data.shape}')

# Analyze EEG data range and scale appropriately
print(f'\nEEG data analysis:')
for i, ch in enumerate(eeg_cols[:3]):  # Show first 3 channels
    ch_data = data[i]
    print(f'  {ch}: min={ch_data.min():.1f}, max={ch_data.max():.1f}, mean={ch_data.mean():.1f}')

# Scale to Volts - this data appears to be in microvolts (typical range -5000 to -1000)
peak = np.nanmax(np.abs(data)) if data.size else 0.0
print(f'Peak amplitude: {peak:.1f}')

if peak > 100.0:  # Definitely in microvolts range
    data *= 1e-6
    print('✓ Scaled data from microvolts to volts (× 1e-6)')
elif peak > 1.0:  # Likely microvolts
    data *= 1e-6
    print('✓ Likely microvolts, scaled to volts (× 1e-6)')
else:
    print('✓ Data appears to be in volts already (no scaling)')

print(f'After scaling - Peak amplitude: {np.nanmax(np.abs(data)):.2e} V')

## Build MNE Raw and preprocess
Creates an MNE RawArray, sets montage and reference, and applies band-pass (and optional notch) filtering.

In [ ]:
# Create Raw
info = mne.create_info(ch_names=eeg_cols, sfreq=sfreq, ch_types=['eeg'] * len(eeg_cols))
raw = mne.io.RawArray(data, info, verbose=False)

# Reference: average reference for ERP
# raw.set_eeg_reference('average', projection=False)

# Filtering
raw.filter(l_freq=l_freq, h_freq=h_freq, fir_design='firwin', verbose=False)
if notch:
    try:
        raw.notch_filter(freqs=[notch], verbose=False)
    except Exception as e:
        print('Notch filter skipped:', e)

raw

## Events, epochs, and ERPs
Converts the event/marker column to MNE Annotations, creates epochs, and plots condition-wise ERPs.

In [19]:
def _extract_events(df, event_col, times, sfreq, event_map_hint=None):
    """Create MNE-compatible events and event_id from an event column.
    Returns events (n,3), event_id dict, and annotations for plotting/timeline.
    """
    if event_col is None:
        return None, None, None
    
    ser = df[event_col]
    print(f'Processing event column "{event_col}"')
    print(f'Event values found: {sorted(ser.unique())}')
    print(f'Total values: {len(ser)}')
    
    # Handle numeric stimulus codes (common in LSL data)
    if pd.api.types.is_numeric_dtype(ser):
        vals = pd.to_numeric(ser, errors='coerce').fillna(0).to_numpy()
        
        # Find unique non-zero values (ignore 0 = no stimulus)
        uniq_codes = sorted([int(u) for u in np.unique(vals) if int(u) != 0])
        print(f'Non-zero stimulus codes: {uniq_codes}')
        
        # Count occurrences of each stimulus code
        for code in uniq_codes:
            count = np.sum(vals == code)
            print(f'  Code {code}: {count} occurrences in data')
        
        # Create event_id mapping
        if event_map_hint and isinstance(event_map_hint, dict):
            # Use provided mapping, validate against actual data
            event_id = {}
            for label, code in event_map_hint.items():
                if code in uniq_codes:
                    event_id[label] = int(code)
                    print(f'  Mapped: {label} -> {code}')
                else:
                    print(f'  Warning: Code {code} for "{label}" not found in data')
            
            # Add any unmapped codes
            mapped_codes = set(event_id.values())
            for code in uniq_codes:
                if code not in mapped_codes:
                    event_id[f'stim_{code}'] = int(code)
                    print(f'  Auto-mapped: stim_{code} -> {code}')
        else:
            # Auto-generate mapping
            event_id = {f'stim_{code}': int(code) for code in uniq_codes}
            print(f'  Auto-generated mapping: {event_id}')
        
        # Extract events: Find stimulus onsets
        # Method 1: Try rising edge detection (0 to stimulus)
        prev = 0
        events_rising = []
        for i, v in enumerate(vals):
            curr = int(v)
            if curr != 0 and prev == 0:  # Rising edge to stimulus
                events_rising.append([i, 0, curr])
            prev = curr
        
        print(f'Rising edge method found {len(events_rising)} events')
        
        # Method 2: Find all stimulus transitions (any change to non-zero)
        events_transitions = []
        prev = 0
        for i, v in enumerate(vals):
            curr = int(v)
            if curr != 0 and curr != prev:  # Any transition to a different stimulus
                events_transitions.append([i, 0, curr])
            prev = curr
        
        print(f'Transition method found {len(events_transitions)} events')
        
        # Method 3: Simple approach - find all non-zero stimulus onsets
        # Look for any change from non-stimulus to stimulus
        events_simple = []
        for i, v in enumerate(vals):
            curr = int(v)
            if curr != 0:  # Current sample has stimulus
                # Check if previous sample was different (either 0 or different stimulus)
                prev_val = int(vals[i-1]) if i > 0 else 0
                if prev_val != curr:  # This is an onset of current stimulus
                    events_simple.append([i, 0, curr])
        
        print(f'Simple onset method found {len(events_simple)} events')
        
        # Choose the method that found the most reasonable number of events
        if len(events_simple) > len(events_rising) and len(events_simple) > 0:
            events = np.array(events_simple, dtype=int)
            print(f'Using simple onset method (most events found)')
        elif len(events_transitions) > len(events_rising) and len(events_transitions) > 0:
            events = np.array(events_transitions, dtype=int)
            print(f'Using transition method')
        elif len(events_rising) > 0:
            events = np.array(events_rising, dtype=int)
            print(f'Using rising edge method')
        else:
            events = None
            print('No events found with any method!')
        
        if events is not None and len(events) > 0:
            print(f'\nFinal result: {len(events)} stimulus events detected')
            
            # Count events by type
            for label, code in event_id.items():
                count = np.sum(events[:, 2] == code)
                print(f'  {label}: {count} events')
            
            # Validate event timing
            if len(events) > 1:
                intervals = np.diff(events[:, 0]) / sfreq  # Convert to seconds
                print(f'  Event intervals: min={intervals.min():.3f}s, max={intervals.max():.3f}s, mean={intervals.mean():.3f}s')
            
            # Create annotations for timeline visualization
            if times is not None:
                onsets = events[:, 0] / sfreq
                # Map codes back to labels
                inv_map = {v: k for k, v in event_id.items()}
                descriptions = [inv_map.get(int(code), f'stim_{int(code)}') for code in events[:, 2]]
                ann = mne.Annotations(onset=onsets, duration=[0]*len(onsets), description=descriptions)
            else:
                ann = None
        else:
            print('No stimulus events detected!')
            ann = None
            
        return events, event_id, ann
        
    else:
        # Handle string/object event labels (legacy code for other data formats)
        labels = ser.fillna('')
        if event_map_hint:
            event_id = {k: int(v) for k, v in event_map_hint.items()}
        else:
            uniq = [u for u in pd.unique(labels) if str(u) != '' and str(u).lower() not in ['0', 'nan', 'none']]
            event_id = {str(u): i+1 for i, u in enumerate(sorted(map(str, uniq)))}
        
        # Build events at samples where label is non-empty
        on_idx = np.where(labels.astype(str).str.len() > 0)[0]
        events = []
        for idx in on_idx:
            desc = str(labels.iloc[idx])
            code = event_id.get(desc)
            if code is None:
                continue
            events.append([int(idx), 0, int(code)])
        events = np.array(events, dtype=int) if events else None
        
        if times is not None and events is not None and len(events):
            onsets = events[:,0] / sfreq
            ann = mne.Annotations(onset=onsets, duration=[0]*len(onsets), description=[ser.iloc[i] for i in events[:,0]])
        else:
            ann = None
            
        return events, event_id, ann

# Extract events from the stimulus column
print("=== EVENT EXTRACTION ===")
events, event_id, ann = _extract_events(df, event_col, times_rel, sfreq, event_map_hint)

# Debug: Show first few events if found
if events is not None and len(events) > 0:
    print(f'\nFirst 10 events:')
    for i in range(min(10, len(events))):
        sample, _, code = events[i]
        time_sec = sample / sfreq
        # Find label for this code
        label = None
        for lbl, cd in event_id.items():
            if cd == code:
                label = lbl
                break
        print(f'  Event {i+1}: sample={sample}, time={time_sec:.3f}s, code={code} ({label})')

if ann is not None:
    raw.set_annotations(ann)
    print(f'\n✓ Added {len(ann)} annotations to raw data')

if events is None or event_id is None or len(events) == 0:
    print('\n❌ No events detected. Please check:')
    print('   - Event column name (event_col_hint)')
    print('   - Event mapping (event_map_hint)')
    print('   - CSV stimulus values')
    
    # Debug info
    if event_col is not None:
        print(f'\nDEBUG INFO:')
        print(f'Event column: {event_col}')
        print(f'First 20 values: {df[event_col].head(20).tolist()}')
        print(f'Value counts: {df[event_col].value_counts().to_dict()}')
else:
    print(f'\n✓ Successfully detected {len(events)} events')
    print(f'Event ID mapping: {event_id}')
    print(f'Ready for epoching!')

=== EVENT EXTRACTION ===
Processing event column "stim"
Event values found: [np.int64(0), np.int64(1), np.int64(2)]
Total values: 40862
Non-zero stimulus codes: [1, 2]
  Code 1: 30333 occurrences in data
  Code 2: 10280 occurrences in data
  Mapped: Standard -> 1
  Mapped: Target -> 2
Rising edge method found 1 events
Transition method found 68 events
Simple onset method found 68 events
Using simple onset method (most events found)

Final result: 68 stimulus events detected
  Standard: 34 events
  Target: 34 events
  Event intervals: min=0.282s, max=7.615s, mean=1.700s

First 10 events:
  Event 1: sample=0, time=0.000s, code=1 (Standard)
  Event 2: sample=229, time=0.653s, code=2 (Target)
  Event 3: sample=485, time=1.383s, code=1 (Standard)
  Event 4: sample=1101, time=3.140s, code=2 (Target)
  Event 5: sample=1425, time=4.064s, code=1 (Standard)
  Event 6: sample=4095, time=11.679s, code=2 (Target)
  Event 7: sample=4291, time=12.238s, code=1 (Standard)
  Event 8: sample=5765, time=1

In [27]:
# Create epochs from events
print("=== EPOCH CREATION ===")

if events is not None and event_id is not None and len(events) > 0:
    # Create epochs using events directly (more reliable for precise timing)
    epochs = mne.Epochs(
        raw, events, event_id=event_id, 
        tmin=tmin, tmax=tmax, baseline=baseline, 
        preload=True, detrend=1, 
        reject_by_annotation=True, verbose=False
    )
    print(f'✓ Created epochs: {epochs}')
    
    # Show epoch statistics
    print(f'\nEpoch statistics:')
    for cond_name, cond_code in event_id.items():
        try:
            if cond_name in epochs.event_id:
                cond_epochs = epochs[cond_name]
                print(f'  {cond_name}: {len(cond_epochs)} epochs')
            else:
                cond_epochs = epochs[cond_code]
                print(f'  {cond_name}: {len(cond_epochs)} epochs')
        except Exception as e:
            print(f'  {cond_name}: Error accessing epochs - {e}')
            
else:
    epochs = None
    print('❌ Cannot create epochs - no valid events found')

=== EPOCH CREATION ===
✓ Created epochs: <Epochs | 67 events (all good), -0.2 – 0.801 s (baseline -0.2 – 0 s), ~2.2 MiB, data loaded,
 'Standard': 33
 'Target': 34>

Epoch statistics:
  Standard: 33 epochs
  Target: 34 epochs


In [28]:
# Plot ERPs per condition
print("=== ERP PLOTTING ===")

if epochs is not None:
    evokeds = {}
    
    for cond_name, cond_code in event_id.items():
        try:
            # Select epochs by condition name or code
            if cond_name in epochs.event_id:
                cond_epochs = epochs[cond_name]
            else:
                cond_epochs = epochs[cond_code]
            
            if len(cond_epochs) > 0:
                ev = cond_epochs.average()
                evokeds[cond_name] = ev
                print(f'✓ {cond_name}: {len(cond_epochs)} epochs averaged')
                
                # Plot individual condition ERP
                ev.plot(spatial_colors=False, time_unit='s', titles=f'ERP: {cond_name}')
            else:
                print(f'⚠ {cond_name}: No epochs found')
                
        except Exception as e:
            print(f'❌ Error processing {cond_name}: {e}')
    
    print(f'\n✓ Generated ERPs for {len(evokeds)} conditions')
    
else:
    evokeds = {}
    print('❌ Cannot plot ERPs - no valid epochs found')

=== ERP PLOTTING ===
✓ Standard: 33 epochs averaged
✓ Target: 34 epochs averaged
✓ Target: 34 epochs averaged

✓ Generated ERPs for 2 conditions

✓ Generated ERPs for 2 conditions


In [29]:
# Difference wave analysis (Target - Standard)
print("=== DIFFERENCE WAVE ANALYSIS ===")

if len(evokeds) >= 2:
    # Find target and standard conditions
    target_key = None
    standard_key = None
    
    for key in evokeds.keys():
        if 'target' in key.lower():
            target_key = key
        elif 'standard' in key.lower():
            standard_key = key
    
    if target_key and standard_key and target_key in evokeds and standard_key in evokeds:
        try:
            # Create difference wave (Target - Standard)
            ev_diff = mne.combine_evoked([evokeds[target_key], evokeds[standard_key]], weights=[1, -1])
            print(f'✓ Created difference wave: {target_key} - {standard_key}')
            
            # Plot difference wave
            ev_diff.plot(spatial_colors=True, time_unit='s', titles='ERP: Target - Standard (Difference Wave)')
            
            # Topographic map around P300 time window (250-450ms)
            try:
                ev_diff.plot_topomap(
                    times=np.linspace(0.25, 0.45, 5), 
                    ch_type='eeg', time_unit='s', 
                    scalings=dict(eeg=1e6),  # Scale to microvolts for display
                    title='Target - Standard Difference'
                )
                print('✓ Generated P300 topographic maps')
            except Exception as e:
                print(f'⚠ Topomap generation failed: {e}')
                
        except Exception as e:
            print(f'❌ Error creating difference wave: {e}')
    else:
        print('⚠ Could not create Target-Standard difference wave')
        print(f'  Target condition found: {target_key}')
        print(f'  Standard condition found: {standard_key}')
        print(f'  Available conditions: {list(evokeds.keys())}')
        
else:
    print('❌ Need at least 2 conditions for difference wave analysis')
    print(f'  Available conditions: {list(evokeds.keys()) if evokeds else "None"}')

=== DIFFERENCE WAVE ANALYSIS ===
✓ Created difference wave: Target - Standard
⚠ Topomap generation failed: Evoked.plot_topomap() got an unexpected keyword argument 'title'


C:\Users\Admin\AppData\Local\Temp\ipykernel_19968\862606817.py:22: RuntimeWarning: Channel locations not available. Disabling spatial colors.
  ev_diff.plot(spatial_colors=True, time_unit='s', titles='ERP: Target - Standard (Difference Wave)')


In [32]:
# Grand Average ERP Plot (Target vs Standard)
print("=== GRAND AVERAGE ERP COMPARISON ===")

if len(evokeds) >= 2:
    # Find target and standard conditions
    target_key = None
    standard_key = None
    
    for key in evokeds.keys():
        if 'target' in key.lower():
            target_key = key
        elif 'standard' in key.lower():
            standard_key = key
    
    if target_key and standard_key and target_key in evokeds and standard_key in evokeds:
        try:
            # Get the evoked responses
            target_evoked = evokeds[target_key]
            standard_evoked = evokeds[standard_key]
            
            # Create a combined plot with both conditions
            print(f'✓ Plotting grand average: {target_key} vs {standard_key}')
            
            # Method 1: Use MNE's compare_evokeds function for elegant overlay
            try:
                from mne.viz import compare_evokeds
                compare_evokeds(
                    evokeds=[target_evoked, standard_evoked],
                    picks='eeg',  # Use all EEG channels
                    colors=['red', 'blue'],
                    linestyles=['-', '-'],
                    legend=[target_key, standard_key],
                    show_sensors=True,
                    time_unit='s',
                    title='Grand Average ERP: Target vs Standard'
                )
                print('✓ Generated MNE compare_evokeds plot')
            except (ImportError, AttributeError) as e:
                print(f'MNE compare_evokeds not available: {e}')
                print('Using alternative plotting methods...')
            
            # Method 2: Grand average as single traces (mean across all channels)
            fig, ax = plt.subplots(figsize=(12, 6))
            
            # Calculate grand average (mean across all channels)
            times = target_evoked.times
            target_data = target_evoked.data.mean(axis=0)  # Average across channels
            standard_data = standard_evoked.data.mean(axis=0)  # Average across channels
            
            # Plot both conditions
            ax.plot(times, target_data * 1e6, 'r-', linewidth=2.5, label=f'{target_key}')
            ax.plot(times, standard_data * 1e6, 'b-', linewidth=2.5, label=f'{standard_key}')
            
            # Add vertical line at stimulus onset
            ax.axvline(0, color='black', linestyle='--', alpha=0.7, linewidth=1)
            ax.text(0.01, ax.get_ylim()[1]*0.9, 'Stimulus\nOnset', fontsize=10, ha='left')
            
            # Formatting
            ax.set_xlabel('Time (s)', fontsize=12)
            ax.set_ylabel('Amplitude (μV)', fontsize=12)
            ax.set_title('Grand Average ERP: Target vs Standard\n(Mean across all channels)', fontsize=14, fontweight='bold')
            ax.legend(fontsize=12)
            ax.grid(True, alpha=0.3)
            
            # Add some ERP component markers
            ax.axvspan(0.08, 0.12, alpha=0.15, color='green')
            ax.text(0.10, ax.get_ylim()[0]*0.8, 'N100', fontsize=9, ha='center', color='darkgreen')
            
            ax.axvspan(0.15, 0.25, alpha=0.15, color='orange')
            ax.text(0.20, ax.get_ylim()[0]*0.8, 'P200', fontsize=9, ha='center', color='darkorange')
            
            ax.axvspan(0.25, 0.45, alpha=0.15, color='purple')
            ax.text(0.35, ax.get_ylim()[0]*0.8, 'P300', fontsize=9, ha='center', color='purple')
            
            plt.tight_layout()
            plt.show()
            
            # Method 3: Side-by-Side comparison with overlaid grand averages
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            
            # Plot Target individual channels + grand average
            target_evoked.plot(
                axes=axes[0], 
                spatial_colors=False, 
                time_unit='s', 
                titles=f'{target_key} ERP',
                show=False,
                gfp=False
            )
            axes[0].plot(times, target_data * 1e6, 'r-', linewidth=3, label='Grand Average')
            axes[0].legend()
            
            # Plot Standard individual channels + grand average
            standard_evoked.plot(
                axes=axes[1], 
                spatial_colors=False, 
                time_unit='s', 
                titles=f'{standard_key} ERP',
                show=False,
                gfp=False
            )
            axes[1].plot(times, standard_data * 1e6, 'b-', linewidth=3, label='Grand Average')
            axes[1].legend()
            
            # Plot direct comparison of grand averages
            axes[2].plot(times, target_data * 1e6, 'r-', linewidth=2.5, label=f'{target_key}')
            axes[2].plot(times, standard_data * 1e6, 'b-', linewidth=2.5, label=f'{standard_key}')
            axes[2].axvline(0, color='black', linestyle='--', alpha=0.7)
            axes[2].set_xlabel('Time (s)')
            axes[2].set_ylabel('Amplitude (μV)')
            axes[2].set_title('Grand Average Comparison')
            axes[2].legend()
            axes[2].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Method 4: Statistical comparison
            fig, ax = plt.subplots(figsize=(12, 6))
            
            # Calculate difference wave
            diff_data = target_data - standard_data
            
            # Plot all three: Target, Standard, and Difference
            ax.plot(times, target_data * 1e6, 'r-', linewidth=2, label=f'{target_key}', alpha=0.8)
            ax.plot(times, standard_data * 1e6, 'b-', linewidth=2, label=f'{standard_key}', alpha=0.8)
            ax.plot(times, diff_data * 1e6, 'g-', linewidth=2.5, label='Difference (Target - Standard)')
            
            # Add zero line
            ax.axhline(0, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
            ax.axvline(0, color='black', linestyle='--', alpha=0.7)
            
            ax.set_xlabel('Time (s)', fontsize=12)
            ax.set_ylabel('Amplitude (μV)', fontsize=12)
            ax.set_title('ERP Comparison with Difference Wave\n(Grand Averages)', fontsize=14, fontweight='bold')
            ax.legend(fontsize=11)
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            print(f'✓ Generated grand average comparison plots')
            print(f'  {target_key}: Peak amplitude = {target_data.max()*1e6:.2f} μV at {times[target_data.argmax()]:.3f}s')
            print(f'  {standard_key}: Peak amplitude = {standard_data.max()*1e6:.2f} μV at {times[standard_data.argmax()]:.3f}s')
            print(f'  Difference peak: {diff_data.max()*1e6:.2f} μV at {times[diff_data.argmax()]:.3f}s')
            
            # P300 analysis (250-450ms window)
            p300_start, p300_end = 0.25, 0.45
            p300_mask = (times >= p300_start) & (times <= p300_end)
            if np.any(p300_mask):
                target_p300 = target_data[p300_mask].max() * 1e6
                standard_p300 = standard_data[p300_mask].max() * 1e6
                p300_diff = target_p300 - standard_p300
                print(f'\nP300 Analysis (250-450ms):')
                print(f'  {target_key} P300: {target_p300:.2f} μV')
                print(f'  {standard_key} P300: {standard_p300:.2f} μV')
                print(f'  P300 difference: {p300_diff:.2f} μV')
            
        except Exception as e:
            print(f'❌ Error creating grand average plots: {e}')
    else:
        print('⚠ Need both Target and Standard conditions for comparison')
        print(f'  Target condition found: {target_key}')
        print(f'  Standard condition found: {standard_key}')
        print(f'  Available conditions: {list(evokeds.keys())}')
        
else:
    print('❌ Need at least 2 conditions for grand average comparison')
    print(f'  Available conditions: {list(evokeds.keys()) if evokeds else "None"}')

=== GRAND AVERAGE ERP COMPARISON ===
✓ Plotting grand average: Target vs Standard
MNE compare_evokeds not available: cannot import name 'compare_evokeds' from 'mne.viz' (d:\projects\EEG-ExPy\.venv\lib\site-packages\mne\viz\__init__.py)
Using alternative plotting methods...
✓ Generated grand average comparison plots
  Target: Peak amplitude = 0.00 μV at 0.713s
  Standard: Peak amplitude = 0.00 μV at 0.781s
  Difference peak: 0.00 μV at 0.719s

P300 Analysis (250-450ms):
  Target P300: 0.00 μV
  Standard P300: 0.00 μV
  P300 difference: -0.00 μV
✓ Generated grand average comparison plots
  Target: Peak amplitude = 0.00 μV at 0.713s
  Standard: Peak amplitude = 0.00 μV at 0.781s
  Difference peak: 0.00 μV at 0.719s

P300 Analysis (250-450ms):
  Target P300: 0.00 μV
  Standard P300: 0.00 μV
  P300 difference: -0.00 μV
